# Reproduksi angka evaluasi tanpa GPU

Notebook ini menjalankan `riset-lanjutan/skrip/jalankan_semua.py`, yang menghitung ulang **semua angka evaluasi naskah** dari berkas yang sudah dirilis:

- set uji dalam dua versi rujukan: `lama` (rujukan asli) dan `terkoreksi` (rujukan Toba set uji kita setelah koreksi penutur asli);
- chrF++ dan BLEU semua sistem, selang kepercayaan, uji beda berpasangan dengan koreksi Holm, dan Tabel 3;
- audit varietas Angkola-Mandailing, analisis ESA, Kendall tau, statistik korpus, dan kovariat.

Pelatihan model **tidak** diulang. Prediksi 22 sistem dibaca dari folder `prediksi/`.

**Cara pakai:** pilih runtime CPU biasa (tidak perlu GPU), lalu jalankan sel dari atas ke bawah. Total sekitar 3 sampai 5 menit.

## 1. Pengaturan

Ubah nilai di bawah bila perlu. `FOLDER_LEMBAR` boleh dikosongkan; isi bila ingin ikut meringkas lembar penutur asli (berkas `C1.xlsx`, `C2.xlsx`, `C3.xlsx`, `C5.xlsx` di Google Drive). `SIMPAN_KE_DRIVE` menyalin hasil ke Drive agar tidak hilang saat runtime berhenti.

In [ ]:
REPO_URL = "https://github.com/FeliksMakarios/batak-toba-korpus.git"
CABANG = "claude/toba-language-corpus-xwdop5"
FOLDER_REPO = "/content/batak-toba-korpus"

FOLDER_LEMBAR = ""          # contoh: "/content/drive/MyDrive/Datasets/btb-eksperimen-mt/lembar_penutur"
SIMPAN_KE_DRIVE = False     # True untuk menyalin hasil ke Drive
FOLDER_DRIVE_HASIL = "/content/drive/MyDrive/Datasets/btb-eksperimen-mt/hasil_v3"
B_HOLM = 10000              # sampel bootstrap uji beda; 10000 sama dengan naskah

## 2. Ambil repositori dan pasang pustaka

In [ ]:
import os, subprocess, sys
if not os.path.exists(FOLDER_REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CABANG, REPO_URL, FOLDER_REPO], check=True)
else:
    subprocess.run(["git", "-C", FOLDER_REPO, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu==2.6.0", "openpyxl"], check=True)
print(subprocess.run(["git", "-C", FOLDER_REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
if FOLDER_LEMBAR or SIMPAN_KE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Jalankan skrip

Keluaran setiap tahap dicetak dengan waktu berjalan. Teks NusaX dan NusaTranslation diunduh dari rilis resmi IndoNLP ke folder cache, tidak ke repositori.

In [ ]:
perintah = [sys.executable, "riset-lanjutan/skrip/jalankan_semua.py", "--cache", "/content/cache_unduh",
            "--keluaran", "/content/hasil_v3", "--B_holm", str(B_HOLM)]
if FOLDER_LEMBAR:
    perintah += ["--lembar", FOLDER_LEMBAR]
proses = subprocess.run(perintah, cwd=FOLDER_REPO, capture_output=True, text=True)
print("\n".join(b for b in proses.stdout.splitlines() if "effective_order" not in b))
if proses.returncode != 0:
    print(proses.stderr[-3000:])
    raise RuntimeError("Skrip gagal; lihat pesan di atas.")

## 4. Verifikasi

Angka versi rujukan lama dicocokkan dengan tabel yang sudah dirilis di `hasil/tabel/`. Semua baris seharusnya `lolos = True`.

In [ ]:
import json
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)
H = "/content/hasil_v3"
verif = pd.read_csv(f"{H}/verifikasi.csv")
display(verif)
print("Lolos:", int(verif.lolos.sum()), "dari", len(verif))

## 5. Ringkasan hasil utama

In [ ]:
ringkas = json.load(open(f"{H}/ringkasan.json"))
for k in ["rujukan_dikoreksi", "koreksi_id2bbc", "holm_lama", "holm_terkoreksi", "holm_keputusan_berubah", "korpus"]:
    print(f"{k}: {ringkas[k]}")

In [ ]:
# Tabel utama chrF++ [IK 95%], rujukan terkoreksi
display(pd.read_csv(f"{H}/tabel_utama_terkoreksi.csv"))

In [ ]:
# Tabel 3: chrF++ menurut bahasa asli, rujukan terkoreksi
t3 = pd.read_csv(f"{H}/tabel3_terkoreksi.csv")
display(t3[["sistem", "arah", "asli_indonesia", "asli_toba", "selisih_toba_minus_indonesia",
            "selisih_ik_bawah", "selisih_ik_atas", "p_dua_sisi"]].round(2))

In [ ]:
# Perubahan chrF++ akibat koreksi rujukan (id->bbc, set uji kita, rata-rata benih)
p = pd.read_csv(f"{H}/perubahan_koreksi_rujukan.csv")
p = p[(p.dimensi == "semua") & (p.arah == "id2bbc")].groupby("grup")[["lama", "terkoreksi", "selisih"]].mean()
display(p.sort_values("terkoreksi", ascending=False).round(2))

In [ ]:
# Kesepakatan metrik dan penilai (Kendall tau-b), kedua versi rujukan
k = pd.read_csv(f"{H}/esa_kendall.csv")
display(k[k.metrik == "chrf"].pivot_table(index=["arah", "cakupan"], columns="rujukan", values="kendall_tau_b"))

In [ ]:
# Keluaran id->bbc bertanda Angkola-Mandailing (%): daftar 32 penanda dan daftar ketat 26 penanda
v = pd.read_csv(f"{H}/varietas_keluaran.csv")
pilih = ["rujukan", "ft_nusatranslation_s42", "ft_kami+nusatranslation_s42", "ft_kami_s42", "madlad3b_zeroshot",
         "llm_gemma3-27b-cpt-gotoai-v3-instruct_few_shot_retrieval"]
display(v[v.sistem.isin(pilih)].pivot_table(index="sistem", columns=["set_uji", "daftar"], values="persen_AM").round(1))

## 6. Simpan hasil (opsional)

In [ ]:
import shutil
if SIMPAN_KE_DRIVE:
    shutil.copytree(H, FOLDER_DRIVE_HASIL, dirs_exist_ok=True)
    print("Disalin ke", FOLDER_DRIVE_HASIL)
else:
    shutil.make_archive("/content/hasil_v3", "zip", H)
    print("Hasil dikemas di /content/hasil_v3.zip (unduh dari panel Files).")